# Revenue-Optimal Dynamic Pricing using Reinforcement Learning
#### Policy Learning in a Competitive and Seasonal Synthetic Market

This project develops a **dynamic pricing intelligence framework** that evolves progressively as market information becomes available. Instead of assuming full observability from the outset, the pricing policy is built through staged learning cycles that reflect how real firms transition from heuristic decision rules to data-driven adaptive control.

The lifecycle integrates **exploratory experimentation, structural demand learning, simulation-based reinforcement, real deployment feedback, and long-term adaptive optimization**.

---

### Phase 0: Exploratory Market Entry

At product launch, the firm has **no historical demand or competitive response data**.  
A controlled exploratory pricing policy is therefore used to stimulate observable market reactions.

- An initial operational price \( p_0 \) defines the starting market positioning.  
- Subsequent prices are generated using a **rule-based exploratory policy** designed to induce price variation and reveal elasticity patterns.  
- When insufficient history exists to compute all model features at time \( t \), the system **maintains the last observed price \( p_{t-1} \)** until the required state variables can be constructed.
- This phase generates the first dataset used to estimate:
  - short-term demand elasticity  
  - demand persistence and smoothing dynamics  
  - early competitive interaction signals  

Seasonality effects are initially specified using **expert priors** rather than statistical estimation.

### Phase 1: Structural Demand Modeling

With initial observations available, the firm estimates **baseline structural components** of the pricing environment.

These include:

- price elasticity regimes  
- demand trend persistence  
- competitor reaction intensity  
- revenue response curvature  

The objective of this phase is not yet optimal pricing, but rather **building a stable economic state representation** that will support downstream policy learning.

### Phase 2: Simulation-Based Policy Pre-Training

Using the estimated structural parameters and domain assumptions, the system generates **synthetic multi-year market trajectories**.  
Within these simulated environments, a rule-based pricing controller produces economically coherent pricing paths.

The resulting state–action dataset is used to **pre-train a recurrent pricing policy offline**, allowing the learning system to internalize realistic strategic behaviors before real market exposure.

This step reduces deployment risk and improves initial policy stability.

### Phase 3: Controlled Real-Market Deployment

The pre-trained dynamic pricing policy is deployed in production during a limited operational horizon (e.g., one quarter).  
During this phase, the system prioritizes:

- safe exploration within bounded pricing adjustments  
- collection of high-quality behavioral demand data  
- validation of simulated structural assumptions  

New observations allow refinement of elasticity estimates and competitor interaction models.

### Phase 4: Iterative Policy Distillation and Reinforcement

At regular decision cycles (e.g., quarterly), the system updates its structural priors using all accumulated historical data.

A new generation of synthetic scenarios is produced reflecting updated market beliefs, and the pricing policy is retrained.  
Over time, the previously deployed policy replaces heuristic rules as the **expert trajectory generator**, enabling progressive policy distillation.

This creates a continuous improvement loop between:

- empirical evidence  
- scenario simulation  
- policy optimization  

### Phase 5: Seasonal Learning and Long-Term Market Adaptation

After approximately one year of operation, sufficient calendar variation exists to statistically estimate **seasonal demand effects and promotional cycles**.

Expert assumptions are gradually replaced by data-driven seasonal parameters.  
The pricing system increasingly behaves as an **adaptive digital twin of the competitive market**, capable of long-horizon strategic positioning and regime-dependent pricing adjustments.

### Phase 6: Autonomous Strategic Pricing Optimization (Mature Stage)

In the mature stage, the pricing agent transitions from imitation-based learning to **forward-looking strategic optimization**.

Key characteristics include:

- endogenous regime detection (price wars vs margin expansion phases)  
- dynamic risk-adjusted revenue maximization  
- adaptive reaction speed based on market volatility  
- continuous policy refinement using rolling scenario generation  

At this point, the pricing system operates as a **self-improving decision engine**, integrating real-time signals with learned structural knowledge to sustain competitive advantage.

In [ ]:
import os
import gc
import math
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

rng = np.random.default_rng()

g = torch.Generator()
g.manual_seed(42)

In [ ]:
class MarketPriors:

    def __init__(
        self,
        alpha=7.6,
        elasticity=1.6,
        cross_elasticity=0.45,
        rho=0.45,
        shock_std=0.18,
        comp_k=0.22,
        comp_noise=0.035,
        unit_cost=55,
        fixed_cost=5000,
        season_amp=None,
        season_sigma=None
    ):
        self.alpha = alpha
        self.elasticity = elasticity
        self.cross_elasticity = cross_elasticity
        self.rho = rho
        self.shock_std = shock_std
        self.comp_k = comp_k
        self.comp_noise = comp_noise
        self.unit_cost = unit_cost
        self.fixed_cost = fixed_cost

        self.season_amp = season_amp
        self.season_sigma = season_sigma

In [ ]:

def get_retail_holiday_spikes(start_date, end_date, s_mothers=10, s_fathers=10, s_blackfriday=15, s_xmas=20):

    years = list(range(start_date.year, end_date.year + 1))
    holidays = []

    for y in years:

        # Mother's day (2nd sunday of may)
        may = pd.date_range(f"{y}-05-01", f"{y}-05-31", freq="D")
        sundays = may[may.weekday == 6]
        mothers_day = sundays[1]

        holidays.append({
            "name": "mothers_day",
            "date": mothers_day,
            "sigma": s_mothers   # medium build-up window
        })

        # Father's day (2nd sunday of august)
        aug = pd.date_range(f"{y}-08-01", f"{y}-08-31", freq="D")
        sundays = aug[aug.weekday == 6]
        fathers_day = sundays[1]

        holidays.append({
            "name": "fathers_day",
            "date": fathers_day,
            "sigma": s_fathers
        })

        # Black friday (last friday of november)
        nov = pd.date_range(f"{y}-11-01", f"{y}-11-30", freq="D")
        fridays = nov[nov.weekday == 4]
        black_friday = fridays[-1]

        holidays.append({
            "name": "black_friday",
            "date": black_friday,
            "sigma": s_blackfriday   
        })

        # Christmas 
        holidays.append({
            "name": "christmas",
            "date": datetime(y, 12, 25),
            "sigma": s_xmas  
        })

    return holidays

def seasonality_signal(dates, holiday_spikes, amp_dict, sigma_dict, saturation_level=2.5):

    t = np.arange(len(dates))
    S = np.zeros(len(dates))

    for h in holiday_spikes:

        name = h["name"]
        center = pd.Timestamp(h["date"])
        idx = np.argmin(np.abs(dates - center))

        A = amp_dict[name]
        sigma = sigma_dict[name]
        bump = A * np.exp(-0.5 * ((t - idx) / sigma) ** 2)
        S += bump

    # saturation
    S = saturation_level * np.tanh(S / saturation_level)
    return S


In [ ]:

def demand_step(df_hist, priors, rng, season_signal):

    t = len(df_hist)

    last_price = df_hist.iloc[-1]["price_agent"]
    last_comp = df_hist.iloc[-1]["price_competitor"]
    last_eta = df_hist.iloc[-1]["demand_shock"]

    # AR(1) demand shock evolution 
    eta = priors.rho * last_eta + rng.normal(0, priors.shock_std)

    # competitor reaction 
    reaction = priors.comp_k * (last_price - last_comp)
    comp_noise = rng.normal(0, priors.comp_noise * last_comp)
    comp_price = max(50.0, last_comp + reaction + comp_noise)

    # structural demand model
    log_mu = (
        priors.alpha
        + season_signal[t]
        - priors.elasticity * np.log(last_price)
        + priors.cross_elasticity * np.log(comp_price)
        + eta
    )

    mu = np.exp(log_mu)
    realized_demand = rng.poisson(mu)

    # revenue
    revenue = (last_price - priors.unit_cost) * realized_demand - priors.fixed_cost

    new_row = {
        "price_agent": last_price,
        "price_competitor": comp_price,
        "realized_demand": realized_demand,
        "demand_shock": eta,
        "revenue": revenue
    }

    return new_row


In [ ]:

def build_features(df):

    features = [
        "rel_price_position",
        "comp_gap_slow",
        "demand_trend_slow",
        "demand_trend_medium",
        "revenue_pressure_slow",
        "price_regime_memory",
        "price_volatility_regime"
    ]

    n = len(df)

    # cold start / missing columns
    if not set(features).issubset(df.columns):

        # relative long-term price positioning vs competitor anchor
        df["rel_price_position"] = (
            df["price_agent"].shift(1)
            / df["price_competitor"].shift(1).rolling(30, min_periods=1).mean()
        )

        # smooth competitive pressure removing tactical noise
        df["comp_gap_slow"] = (
            df["price_competitor"].shift(1).rolling(14, min_periods=1).mean()
            - df["price_agent"].shift(1).rolling(14, min_periods=1).mean()
        ) / df["price_agent"].shift(1)

        # structural demand cycle capturing macro consumption regime
        slow_demand = df["realized_demand"].shift(1).rolling(60, min_periods=1).mean()
        df["demand_trend_slow"] = (
            slow_demand
            / slow_demand.rolling(120, min_periods=1).mean()
            - 1
        )

        # medium demand acceleration signal for tactical stance
        df["demand_trend_medium"] = (
            df["realized_demand"].shift(1).rolling(21, min_periods=1).mean()
            / df["realized_demand"].shift(1).rolling(60, min_periods=1).mean()
            - 1
        )

        # long profitability pressure guiding margin expansion risk
        df["revenue_pressure_slow"] = np.tanh(
            df["revenue"].shift(1).rolling(45, min_periods=1).sum() / 80000
        )

        # deviation from recent equilibrium band controlling overshoot
        df["price_regime_memory"] = np.log(
            df["price_agent"].shift(1)
            / df["price_agent"].shift(1).rolling(30, min_periods=1).mean()
        )

        # short-term instability regime moderating adjustment aggressiveness
        df["price_volatility_regime"] = (
            np.log(
                df["price_agent"].shift(1)
                / df["price_agent"].shift(2)
            )
            .rolling(14, min_periods=1)
            .std()
        )

        return df, features

    # incremental update
    i = n - 1

    # relative competitive positioning anchor
    df.loc[i, "rel_price_position"] = (
        df.loc[i-1, "price_agent"]
        / df["price_competitor"].iloc[max(0, i-30):i].mean()
    )

    # persistent competitive mispricing pressure
    comp_mean = df["price_competitor"].iloc[max(0, i-14):i].mean()
    agent_mean = df["price_agent"].iloc[max(0, i-14):i].mean()
    df.loc[i, "comp_gap_slow"] = (
        (comp_mean - agent_mean)
        / df.loc[i-1, "price_agent"]
    )

    # slow demand regime shift
    slow_demand = df["realized_demand"].iloc[max(0, i-60):i].mean()
    slow_demand_ref = df["realized_demand"].iloc[max(0, i-120):i].mean()
    df.loc[i, "demand_trend_slow"] = (
        slow_demand / slow_demand_ref - 1
        if slow_demand_ref > 0 else 0.0
    )

    # medium demand tactical drift
    demand_21 = df["realized_demand"].iloc[max(0, i-21):i].mean()
    demand_60 = df["realized_demand"].iloc[max(0, i-60):i].mean()
    df.loc[i, "demand_trend_medium"] = (
        demand_21 / demand_60 - 1
        if demand_60 > 0 else 0.0
    )

    # profitability regime saturation signal
    rev_sum = df["revenue"].iloc[max(0, i-45):i].sum()
    df.loc[i, "revenue_pressure_slow"] = np.tanh(rev_sum / 80000)

    # price inertia / overshooting detector
    price_mean = df["price_agent"].iloc[max(0, i-30):i].mean()
    df.loc[i, "price_regime_memory"] = (
        np.log(df.loc[i-1, "price_agent"] / price_mean)
        if price_mean > 0 else 0.0
    )

    # local volatility state reducing optimal move size
    returns = np.log(
        df["price_agent"].iloc[max(1, i-14):i]
        / df["price_agent"].iloc[max(0, i-15):i-1]
    )
    df.loc[i, "price_volatility_regime"] = (
        returns.std() if len(returns) > 1 else 0.0
    )

    return df, features

In [ ]:

def rule_based_policy(df, price_min=60, price_max=140):

    row = df.iloc[-1]

    # ===== control parameters (economically calibrated) =====
    signal_strength = 0.028      # ↑ global move size
    noise_std = 0.018            # slight realism boost

    comp_weight = 0.50           # stronger competitor tracking
    demand_weight = 0.18         # more tactical demand sensitivity
    macro_demand_weight = 0.22   # preserve slow cycle

    profit_weight = 0.12         # slightly reduced strategic push
    inertia_weight = 0.16        # ↓ damping → more oscillation

    # ===== competitive anchoring drift =====
    comp_gap = row["comp_gap_slow"]
    drift = comp_weight * comp_gap

    # ===== structural demand regime =====
    drift += macro_demand_weight * row["demand_trend_slow"]

    # ===== medium tactical demand =====
    drift += demand_weight * row["demand_trend_medium"]

    # ===== profitability pressure =====
    drift += profit_weight * row["revenue_pressure_slow"]

    # ===== price inertia stabilizer (VERY important) =====
    drift -= inertia_weight * row["price_regime_memory"]

    # ===== stochastic market shocks =====
    noise = np.random.normal(0, noise_std)

    # ===== bounded diffusion step =====
    log_step = signal_strength * drift + noise
    log_step = 0.10 * np.tanh(log_step / 0.10)

    new_price = row["price_agent"] * np.exp(log_step)
    new_price = np.clip(new_price, price_min, price_max)

    return float(new_price)

In [ ]:

def simulate(start_date, n_days, p0, priors, rng):

    dates = pd.date_range(start_date, periods=n_days, freq="D")

    holiday_spikes = get_retail_holiday_spikes(
        dates.min(), dates.max()
    )

    season_signal = seasonality_signal(
        dates,
        holiday_spikes,
        priors.season_amp,
        priors.season_sigma
    )

    df = pd.DataFrame(index=range(n_days))
    df["date"] = dates

    # ===== initial state =====
    df.loc[0, "price_agent"] = p0
    df.loc[0, "price_competitor"] = p0 * 1.02
    df.loc[0, "demand_shock"] = rng.normal(
        0,
        priors.shock_std / np.sqrt(1 - priors.rho**2)
    )

    # initial demand
    first_log_mu = (
        priors.alpha
        + season_signal[0]
        - priors.elasticity * np.log(p0)
        + priors.cross_elasticity * np.log(df.loc[0, "price_competitor"])
        + df.loc[0, "demand_shock"]
    )

    mu0 = np.exp(first_log_mu)
    df.loc[0, "realized_demand"] = rng.poisson(mu0)
    df.loc[0, "revenue"] = (
        (p0 - priors.unit_cost) * df.loc[0, "realized_demand"]
        - priors.fixed_cost
    )

    # ===== simulation =====
    for t in range(1, n_days):

        df_hist = df.iloc[:t]

        # pricing decision
        if t < 40:
            new_price = df.loc[t-1, "price_agent"]
        else:
            df_feat, _ = build_features(df_hist.copy())
            new_price = rule_based_policy(df_feat)

        df.loc[t, "price_agent"] = new_price

        # environment transition
        new_row = demand_step(df.iloc[:t].copy(), priors, rng, season_signal)

        df.loc[t, "price_competitor"] = new_row["price_competitor"]
        df.loc[t, "realized_demand"] = new_row["realized_demand"]
        df.loc[t, "demand_shock"] = new_row["demand_shock"]
        df.loc[t, "revenue"] = new_row["revenue"]

    return df